# Setup

In [1]:
!pip install -q "datasets<4.0.0" huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 1.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 1.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 1.5 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [2]:
import os
import random
import time
from io import BytesIO
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
import torch
from PIL import Image
from tqdm import tqdm

from datasets import (
    Dataset,
    load_dataset,
    concatenate_datasets,
    ClassLabel,
)
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

In [3]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [5]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


## Configuration

In [32]:
CLOTHING_SAMPLES = 5000
ACCESSORIES_SAMPLES = 3333
ELECTRONICS_SAMPLES = 3333
HF_REPO_ID = "sdmikhalin/ecommerce-10k"
FASHION_DATASET = "ashraq/fashion-product-images-small"
AMAZON_DATASET = "McAuley-Lab/Amazon-Reviews-2023"

## Helper functions

In [8]:
def is_valid_text(example):
    text = example.get("productDisplayName", "")
    if not text or not isinstance(text, str):
        return False
    words = text.split()
    return 3 <= len(words) <= 20

In [9]:
def is_valid_image(example):
    img = example.get("image")
    if img is None:
        return False
    return img.size[0] >= 32 and img.size[1] >= 32

# Clothing Dataset

In [10]:
fashion_raw = load_dataset(FASHION_DATASET, split="train")
fashion = fashion_raw.filter(lambda x: x["masterCategory"] == "Apparel")
fashion = fashion.filter(lambda x: is_valid_text(x) and is_valid_image(x))

top_subcats = [cat for cat, _ in Counter(fashion["subCategory"]).most_common(5)]
samples_per_subcat = CLOTHING_SAMPLES // len(top_subcats)

balanced_items = []
for subcat in top_subcats:
    subset = fashion.filter(lambda x, sc=subcat: x["subCategory"] == sc)
    n_take = min(samples_per_subcat, len(subset))
    subset = subset.shuffle(seed=RANDOM_SEED).select(range(n_take))
    for item in subset:
        balanced_items.append(item)


def format_fashion(example):
    return {
        "image": example["image"].convert("RGB"),
        "category": "clothing",
        "text": example["productDisplayName"].strip(),
    }


fashion = Dataset.from_list(balanced_items)
fashion = fashion.map(format_fashion, remove_columns=fashion.column_names)
fashion = fashion.shuffle(seed=RANDOM_SEED)
print(f"Clothing: {len(fashion)} samples")

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…):   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00001-of-00002-bb459e5ac5f01e(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Filter:   0%|          | 0/44072 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/21355 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21355 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21355 [00:00<?, ? examples/s]

Map:   0%|          | 0/3942 [00:00<?, ? examples/s]

Clothing: 3942 samples


# Accessories Dataset

In [33]:
accessories = fashion_raw.filter(lambda x: x["masterCategory"] == "Accessories")
accessories = accessories.filter(lambda x: is_valid_text(x) and is_valid_image(x))

TOP_ACCESSORIES_SUBCATS = ["Watches", "Bags", "Sunglasses", "Wallets", "Belts"]
samples_per_subcat = ACCESSORIES_SAMPLES // len(TOP_ACCESSORIES_SUBCATS)

balanced_items = []
for subcat in TOP_ACCESSORIES_SUBCATS:
    subset = accessories.filter(lambda x, sc=subcat: x["subCategory"] == sc)
    if len(subset) == 0:
        continue
    n_take = min(samples_per_subcat, len(subset))
    subset = subset.shuffle(seed=RANDOM_SEED).select(range(n_take))
    for item in subset:
        balanced_items.append(item)


def format_accessories(example):
    return {
        "image": example["image"].convert("RGB"),
        "category": "accessories",
        "text": example["productDisplayName"].strip(),
        
    }

accessories = Dataset.from_list(balanced_items)
accessories = accessories.map(format_accessories, remove_columns=accessories.column_names)
accessories = accessories.shuffle(seed=RANDOM_SEED)
print(f"Accessories: {len(accessories)} samples")

Map:   0%|          | 0/2664 [00:00<?, ? examples/s]

Accessories: 2664 samples


# Electronics Dataset

In [16]:
parquet_dfs = []
for shard_idx in range(3):
    path = hf_hub_download(
        repo_id=AMAZON_DATASET,
        filename=f"raw_meta_Electronics/full-{shard_idx:05d}-of-00010.parquet",
        repo_type="dataset",
    )
    parquet_dfs.append(pq.read_table(path).to_pandas())

df = pd.concat(parquet_dfs, ignore_index=True)


def extract_url(images_field):
    try:
        if isinstance(images_field, dict):
            large = images_field.get("large", []) or images_field.get("hi_res", [])
            urls = [u for u in large if u]
            return urls[0] if urls else None
        elif hasattr(images_field, "__len__") and len(images_field) > 0:
            first = images_field[0]
            if isinstance(first, dict):
                return first.get("large") or first.get("hi_res")
            elif isinstance(first, str):
                return first
    except Exception:
        pass
    return None


df["image_url"] = df["images"].apply(extract_url)
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["title_len"] = df["title"].str.split().str.len()

SPAM_MARKERS = [
    "free shipping", "best seller", "best deal", "!!!",
    "limited time", "amazon's choice", "lifetime warranty",
    "pack of", "set of 2", "buy now",
]


def is_spam(title):
    t = title.lower()
    return any(marker in t for marker in SPAM_MARKERS)


valid = df[
    (df["image_url"].notna()) &
    (df["title_len"] >= 5) &
    (df["title_len"] <= 20) &
    (~df["title"].apply(is_spam))
].reset_index(drop=True)

OVERSAMPLE_FACTOR = 2
valid_sample = valid.sample(
    min(ELECTRONICS_SAMPLES * OVERSAMPLE_FACTOR, len(valid)),
    random_state=RANDOM_SEED,
).reset_index(drop=True)


def fetch_image(row):
    try:
        r = requests.get(
            row["image_url"],
            timeout=15,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        if r.status_code != 200:
            return None
        img = Image.open(BytesIO(r.content)).convert("RGB")
        if img.size[0] < 64 or img.size[1] < 64:
            return None
        return {
            "image": img,
            "category": "electronics",
            "text": row["title"].strip(),
            "_title_key": row["title"].lower()[:50],
        }
    except Exception:
        return None


electronics_items = []
seen_titles = set()
rows = valid_sample.to_dict("records")

with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {executor.submit(fetch_image, row): row for row in rows}
    with tqdm(total=ELECTRONICS_SAMPLES, desc="Downloading electronics") as pbar:
        for f in as_completed(futures):
            r = f.result()
            if r is None:
                continue
            if r["_title_key"] in seen_titles:
                continue
            seen_titles.add(r["_title_key"])
            electronics_items.append(r)
            pbar.update(1)
            if len(electronics_items) >= ELECTRONICS_SAMPLES:
                for future in futures:
                    future.cancel()
                break

electronics_items = [
    {k: v for k, v in item.items() if not k.startswith("_")}
    for item in electronics_items
]

electronics = Dataset.from_list(electronics_items)
print(f"Electronics: {len(electronics)} samples")

/tmp/ipykernel_57/3877144898.py:16: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  large = images_field.get("large", []) or images_field.get("hi_res", [])


Electronics: 3333 samples


# Combined and stratified split

In [17]:
combined = concatenate_datasets([fashion, electronics, accessories])
unique_cats = sorted(set(combined["category"]))

combined_labeled = combined.cast_column("category", ClassLabel(names=unique_cats))
split = combined_labeled.train_test_split(
    test_size=0.05,
    seed=RANDOM_SEED,
    stratify_by_column="category",
)

Casting the dataset:   0%|          | 0/11011 [00:00<?, ? examples/s]

In [18]:
def restore_string_category(dataset, names):
    cat_strings = [names[idx] for idx in dataset["category"]]
    dataset = dataset.remove_columns(["category"])
    dataset = dataset.add_column("category", cat_strings)
    return dataset


train_data = restore_string_category(split["train"], unique_cats).shuffle(seed=RANDOM_SEED)
test_data = restore_string_category(split["test"], unique_cats).shuffle(seed=RANDOM_SEED)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")
print(f"Train distribution: {pd.Series(train_data['category']).value_counts().to_dict()}")
print(f"Test distribution: {pd.Series(test_data['category']).value_counts().to_dict()}")

Flattening the indices:   0%|          | 0/10460 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/551 [00:00<?, ? examples/s]

Train: 10460, Test: 551
Train distribution: {'clothing': 3745, 'accessories': 3549, 'electronics': 3166}
Test distribution: {'clothing': 197, 'accessories': 187, 'electronics': 167}


# Publish to HF Hub

In [ ]:
train_data.push_to_hub(HF_REPO_ID, split="train", private=False)
test_data.push_to_hub(HF_REPO_ID, split="test", private=False)
print(f"Published: https://huggingface.co/datasets/{HF_REPO_ID}")